# Show And Download Qwen Solver LoRA Results

Run this notebook after `colab_lora_sft.ipynb` finishes training. It summarizes the saved adapter, plots training metrics when available, runs a few sample generations, and downloads or copies the result folder.

## Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## Locate `new_solver`

In [ ]:
from pathlib import Path
import os

candidates = [
    Path.cwd(),
    Path('/content/new_solver'),
    Path('/content/MathSolverLLMs/new_solver'),
    Path('/content/drive/MyDrive/MathSolverLLMs/new_solver'),
]

project_dir = None
for candidate in candidates:
    if (candidate / 'train_lora_sft.py').exists() and (candidate / 'requirements.txt').exists():
        project_dir = candidate
        break

if project_dir is None:
    raise FileNotFoundError('Could not find new_solver. Set project_dir manually in this cell.')

os.chdir(project_dir)
print('Working directory:', Path.cwd())
!ls -la


## Install Dependencies

In [ ]:
!pip install -q -r requirements.txt matplotlib pandas


## Set Result Paths

In [ ]:
from pathlib import Path

BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
ADAPTER_DIR = Path('outputs/qwen2.5-0.5b-solver-lora')
TEST_FILE = Path('data/test_sft.jsonl')

if not ADAPTER_DIR.exists():
    raise FileNotFoundError(f'Missing adapter directory: {ADAPTER_DIR.resolve()}')
if not TEST_FILE.exists():
    raise FileNotFoundError(f'Missing test file: {TEST_FILE.resolve()}')

print('Base model:', BASE_MODEL)
print('Adapter:', ADAPTER_DIR.resolve())
print('Test file:', TEST_FILE.resolve())


## Inspect Saved Files

In [ ]:
!find outputs/qwen2.5-0.5b-solver-lora -maxdepth 2 -type f -printf '%p\t%k KB\n' | sort


In [ ]:
import json

for name in ['adapter_config.json', 'trainer_state.json', 'training_args.bin']:
    path = ADAPTER_DIR / name
    print(name, 'exists=' + str(path.exists()))

adapter_config = ADAPTER_DIR / 'adapter_config.json'
if adapter_config.exists():
    with adapter_config.open() as f:
        cfg = json.load(f)
    for key in ['base_model_name_or_path', 'r', 'lora_alpha', 'lora_dropout', 'target_modules']:
        print(f'{key}:', cfg.get(key))


## Plot Training Metrics

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

state_path = ADAPTER_DIR / 'trainer_state.json'
if not state_path.exists():
    print('No trainer_state.json found. Skipping plots.')
else:
    with state_path.open() as f:
        state = json.load(f)
    logs = pd.DataFrame(state.get('log_history', []))
    display(logs.tail(20))

    plt.figure(figsize=(10, 5))
    if 'loss' in logs:
        train_logs = logs.dropna(subset=['loss'])
        plt.plot(train_logs['step'], train_logs['loss'], label='train loss')
    if 'eval_loss' in logs:
        eval_logs = logs.dropna(subset=['eval_loss'])
        plt.plot(eval_logs['step'], eval_logs['eval_loss'], label='eval loss')
    plt.xlabel('step')
    plt.ylabel('loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


## Load Adapter And Generate Samples

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print('Loaded adapter on device:', next(model.parameters()).device)


In [ ]:
import json
import random

def read_jsonl(path):
    rows = []
    with path.open() as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

test_rows = read_jsonl(TEST_FILE)
sample_rows = random.Random(7).sample(test_rows, k=min(3, len(test_rows)))

def generate_solution(problem, max_new_tokens=512):
    messages = [
        {'role': 'system', 'content': 'You are a careful mathematical problem solver. Provide a complete solution with all necessary reasoning.'},
        {'role': 'user', 'content': f'Solve the following problem:\n\n{problem}'},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0, inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

for i, row in enumerate(sample_rows, 1):
    print('=' * 100)
    print('Sample', i, 'source=', row.get('source'), 'id=', row.get('id'))
    print('\nPROBLEM:\n', row['problem'][:1200])
    print('\nREFERENCE FINAL ANSWER:', row.get('final_answer') or '<none>')
    print('\nMODEL OUTPUT:\n', generate_solution(row['problem']))


## Save Sample Outputs

In [ ]:
pred_path = ADAPTER_DIR / 'sample_generations.jsonl'
with pred_path.open('w', encoding='utf-8') as f:
    for row in sample_rows:
        pred = generate_solution(row['problem'])
        f.write(json.dumps({
            'id': row.get('id'),
            'source': row.get('source'),
            'problem': row.get('problem'),
            'final_answer': row.get('final_answer'),
            'prediction': pred,
        }, ensure_ascii=False) + '\n')
print('Wrote', pred_path)


## Download Or Copy Results

In [ ]:
ZIP_NAME = 'qwen2.5-0.5b-solver-lora-results.zip'
!zip -r {ZIP_NAME} outputs/qwen2.5-0.5b-solver-lora

from google.colab import files
files.download(ZIP_NAME)


In [ ]:
# Optional: copy the full result folder and zip to Google Drive.
!mkdir -p /content/drive/MyDrive/new_solver_outputs
!cp -r outputs/qwen2.5-0.5b-solver-lora /content/drive/MyDrive/new_solver_outputs/
!cp qwen2.5-0.5b-solver-lora-results.zip /content/drive/MyDrive/new_solver_outputs/
